In [ ]:
import gc
import os
import random
import joblib
import pickle
import itertools
import warnings
import scipy as sp
import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
import seaborn as sns
import matplotlib.pyplot as plt
from glob import glob
from pathlib import Path
from lightgbm import LGBMClassifier
from tqdm.auto import tqdm
from sklearn.model_selection import StratifiedKFold, train_test_split, GroupKFold
from sklearn.metrics import log_loss, roc_auc_score, matthews_corrcoef, f1_score
from sklearn.preprocessing import LabelEncoder,StandardScaler
from catboost import Pool, CatBoostRegressor, CatBoostClassifier
warnings.simplefilter('ignore')

In [ ]:
class CFG:
    VER = 1
    AUTHOR = 'William'
    COMPETITION = 'icr-identify-age-related-conditions'
    DATA_PATH = Path('/kaggle/input/icr-identify-age-related-conditions')
    OOF_DATA_PATH = Path('./oof')
    MODEL_DATA_PATH = Path('./models')
    #METHOD_LIST = ['lightgbm', 'xgboost', 'catboost']
    METHOD_LIST = ['lightgbm']
    seed = 42 #3407 #52
    n_folds = 10 #replaced 20
    target_col = 'Class'
    metric = 'balanced_log_loss'
    metric_maximize_flag = False
    num_boost_round = 1000
    early_stopping_round = 300
    verbose = 200
    boosting_type = 'dart'
    lgb_params = {
        'objective': 'multiclass', # 'binary', 'multiclass'
        'metric': 'multi_logloss', # 'auc', 'multi_logloss'
        'num_class': 4,
        'boosting': boosting_type,
        'device_type':'gpu',
        'learning_rate': 1e-4, #0.005,
        'num_leaves': 5,
        'feature_fraction': 0.50,
        'bagging_fraction': 0.80,
        'lambda_l1': 2, 
        'lambda_l2': 4,
        'n_jobs': -1,
        'is_unbalance':True, #added balancing
        'verbose': -1, #added silence
        # 'min_data_in_leaf': 40,
        # 'bagging_freq': 10,
        'seed': seed,
    }
#     xgb_params = {
#         'objective':'multi:softprob', # 'binary:logistic',
#         'eval_metric': 'mlogloss',
#         'learning_rate': 0.005, 
#         'max_depth': 4,
#         'colsample_bytree': 0.50,
#         'subsample': 0.80,
# #         'eta': 0.03,
#         'gamma': 1.5,
#         'num_class': 4,
#         # 'lambda': 70,
#         # 'min_child_weight': 8,
#         # 'eval_metric':'logloss',
#         # 'tree_method': 'gpu_hist',
#         # 'predictor':'gpu_predictor',
#         'random_state': seed,
#     }
    
#     cat_params = {
#         'learning_rate': 0.005, 
#         'iterations': num_boost_round, 
#         'depth': 4, # 
#         'colsample_bylevel': 0.50,
# #         'subsample': 0.80,
#         'l2_leaf_reg': 3, # 3, 30
#         'random_seed': seed,
#         'auto_class_weights': 'Balanced',
#         'loss_function':'MultiClass',
#         #'task_type':'GPU'
#     }
    

In [ ]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
seed_everything(CFG.seed)

In [ ]:
!mkdir oof
!mkdir models

# Load Data

In [ ]:
train = pd.read_csv(CFG.DATA_PATH / 'train.csv')
test_df = pd.read_csv(CFG.DATA_PATH / 'test.csv')
greeks_df = pd.read_csv(CFG.DATA_PATH / 'greeks.csv')
submission_df = pd.read_csv(CFG.DATA_PATH / 'sample_submission.csv')

merged_df = pd.merge(train, greeks_df, on='Id').drop(columns=['Class','Beta','Gamma','Delta','Epsilon'], axis=1)
train_df = merged_df.copy()
greek_str2int = {}
greek_str2int['Alpha'] = {'A': 0, 'B': 1,'D': 2, 'G': 3}
train_df['Alpha'] = train_df['Alpha'].map(greek_str2int['Alpha'])
train_df.rename(columns={'Alpha': 'Class'}, inplace=True)
train_df['Class'] = train_df['Class'].astype('float64')

test_df[CFG.target_col] = -1
all_df = pd.concat([train_df, test_df])

## Metrics

In [ ]:
def competition_log_loss(y_true, y_pred):
    N_0 = np.sum(1 - y_true)
    N_1 = np.sum(y_true)
    p_1 = np.clip(y_pred, 1e-15, 1 - 1e-15)
    p_0 = 1 - p_1
    log_loss_0 = -np.sum((1 - y_true) * np.log(p_0)) / N_0
    log_loss_1 = -np.sum(y_true * np.log(p_1)) / N_1
    return (log_loss_0 + log_loss_1)/2

def balanced_log_loss(y_true, y_pred):
    N_0 = np.sum(1 - y_true)
    N_1 = np.sum(y_true)
    p_1 = np.clip(y_pred, 1e-15, 1 - 1e-15)
    p_0 = 1 - p_1
    log_loss_0 = -np.sum((1 - y_true) * np.log(p_0))
    log_loss_1 = -np.sum(y_true * np.log(p_1))
    w_0 = 1 / N_0
    w_1 = 1 / N_1
    balanced_log_loss = 2*(w_0 * log_loss_0 + w_1 * log_loss_1) / (w_0 + w_1)
    return balanced_log_loss/(N_0+N_1)

In [ ]:
def lgb_metric(y_true, y_pred):
    return 'balanced_log_loss', balanced_log_loss(y_true, y_pred), False

In [ ]:
def calc_log_loss_weight(y_true):
    nc = np.bincount(y_true)
    w0, w1 = 1/(nc[0]/y_true.shape[0]), 1/(nc[1]/y_true.shape[0])
    return w0, w1

def lightgbm_training(x_train: pd.DataFrame, y_train: pd.DataFrame, x_valid: pd.DataFrame, y_valid: pd.DataFrame, features: list, categorical_features: list):
#     train_w0, train_w1 = calc_log_loss_weight(y_train)
#     valid_w0, valid_w1 = calc_log_loss_weight(y_valid)
#     lgb_train = lgb.Dataset(x_train, y_train, weight=y_train.map({0: train_w0, 1: train_w1, 2: train_w1, 3: train_w1}), categorical_feature=categorical_features)
#     lgb_valid = lgb.Dataset(x_valid, y_valid, weight=y_valid.map({0: valid_w0, 1: valid_w1, 2: train_w1, 3: train_w1}), categorical_feature=categorical_features)

    lgb_train = lgb.Dataset(x_train, y_train, categorical_feature=categorical_features)
    lgb_valid = lgb.Dataset(x_valid, y_valid, categorical_feature=categorical_features)
    model = lgb.train(
                params = CFG.lgb_params,
                train_set = lgb_train,
                num_boost_round = CFG.num_boost_round,
                valid_sets = [lgb_train, lgb_valid],
                early_stopping_rounds = CFG.early_stopping_round,
                verbose_eval = CFG.verbose,
                # feval = lgb_metric,
            )
    # Predict validation
    valid_pred = model.predict(x_valid)
    return model, valid_pred

# def xgboost_training(x_train: pd.DataFrame, y_train: pd.DataFrame, x_valid: pd.DataFrame, y_valid: pd.DataFrame, features: list, categorical_features: list):
#     train_w0, train_w1 = calc_log_loss_weight(y_train)
#     valid_w0, valid_w1 = calc_log_loss_weight(y_valid)
#     xgb_train = xgb.DMatrix(data=x_train, label=y_train, weight=y_train.map({0: train_w0, 1: train_w1, 2: train_w1, 3: train_w1}))
#     xgb_valid = xgb.DMatrix(data=x_valid, label=y_valid, weight=y_valid.map({0: valid_w0, 1: valid_w1, 2: valid_w1, 3: valid_w1}))
#     model = xgb.train(
#                 CFG.xgb_params, 
#                 dtrain = xgb_train, 
#                 num_boost_round = CFG.num_boost_round, 
#                 evals = [(xgb_train, 'train'), (xgb_valid, 'eval')], 
#                 early_stopping_rounds = CFG.early_stopping_round, 
#                 verbose_eval = CFG.verbose,
#                 # feval = xgb_metric, 
#                 # maximize = CFG.metric_maximize_flag, 
#             )
#     # Predict validation
#     valid_pred = model.predict(xgb.DMatrix(x_valid), iteration_range=(0, model.best_ntree_limit))
#     return model, valid_pred
    
# def catboost_training(x_train: pd.DataFrame, y_train: pd.DataFrame, x_valid: pd.DataFrame, y_valid: pd.DataFrame, features: list, categorical_features: list):
#     train_w0, train_w1 = calc_log_loss_weight(y_train)
#     valid_w0, valid_w1 = calc_log_loss_weight(y_valid)
#     cat_train = Pool(data=x_train, label=y_train, weight=y_train.map({0: train_w0, 1: train_w1, 2: train_w1, 3: train_w1}), cat_features=categorical_features)
#     cat_valid = Pool(data=x_valid, label=y_valid, weight=y_valid.map({0: valid_w0, 1: valid_w1, 2: valid_w1, 3: valid_w1}), cat_features=categorical_features)
#     model = CatBoostClassifier(**CFG.cat_params) # , eval_metric = CatboostMetric
#     model.fit(cat_train, 
#               eval_set=[cat_valid],
#               early_stopping_rounds=CFG.early_stopping_round, 
#               verbose=CFG.verbose, 
#               use_best_model=True)
#     # Predict validation
#     valid_pred = model.predict_proba(x_valid)[:, 1]
#     return model, valid_pred

def gradient_boosting_model_cv_training(method: str, train_df: pd.DataFrame, features: list, categorical_features: list):
    # Create a numpy array to store out of folds predictions
    oof_predictions = np.zeros([len(train_df),4])
    oof_fold = np.zeros([len(train_df),4])
    kfold = StratifiedKFold(n_splits = CFG.n_folds, shuffle = True, random_state = CFG.seed)
    for fold, (train_index, valid_index) in enumerate(kfold.split(train_df, train_df[CFG.target_col])):
        print('-'*50)
        print(f'{method} training fold {fold + 1}')
        
        x_train = train_df[features].iloc[train_index]
        y_train = train_df[CFG.target_col].iloc[train_index]
        x_valid = train_df[features].iloc[valid_index]
        y_valid = train_df[CFG.target_col].iloc[valid_index]
        if method == 'lightgbm':
            model, valid_pred = lightgbm_training(x_train, y_train, x_valid, y_valid, features, categorical_features)
        if method == 'xgboost':
            model, valid_pred = xgboost_training(x_train, y_train, x_valid, y_valid, features, categorical_features)
        if method == 'catboost':
            model, valid_pred = catboost_training(x_train, y_train, x_valid, y_valid, features, categorical_features)

        logloss = log_loss(y_valid, valid_pred)
        competition_logloss = competition_log_loss(y_valid, valid_pred[:,1])
        balanced_logloss = balanced_log_loss(y_valid, valid_pred[:, 1])
        print(f"Fold: {fold+1}, log loss: {round(logloss, 3)}, competition log loss: {round(competition_logloss, 3)}, balanced los loss: {round(balanced_logloss, 3)}")
        # Save best model
        pickle.dump(model, open(CFG.MODEL_DATA_PATH / f'{method}_fold{fold + 1}_seed{CFG.seed}_ver{CFG.VER}.pkl', 'wb'))
        del x_train, x_valid, y_train, y_valid, model, valid_pred
        gc.collect()


In [ ]:
numerical_features = ['AB', 'AF', 'AH', 'AM', 'AR', 'AX', 'AY', 'AZ', 'BC', 
                      'BD', 'BN', 'BP', 'BQ', 'BR', 'BZ',
                      'CB', 'CC', 'CD', 'CF', 'CH', 'CL', 
                      'CR', 'CS', 'CU', 'CW',
                      'DA', 'DE', 'DF', 'DH', 'DI', 'DL', 'DN', 'DU', 'DV', 'DY',
                      'EB', 'EE', 'EG', 'EH', 'EL', 'EP', 'EU',
                      'FC', 'FD', 'FE', 'FI', 'FL', 'FR', 'FS',
                      'GB', 'GE', 'GF', 'GH', 'GI', 'GL']
categorical_features = ['EJ']
features = numerical_features + categorical_features
str2int_dict = {}
str2int_dict['EJ'] = {'A': 1, 'B': 2}


In [ ]:
def Preprocessing(input_df: pd.DataFrame):
    input_df = input_df.rename(columns={'BD ': 'BD', 'CD ': 'CD', 'CW ': 'CW', 'FD ': 'FD'})
    train_df = input_df[input_df[CFG.target_col] != -1].copy()
    test_df = input_df[input_df[CFG.target_col] == -1].copy()
    sc = StandardScaler()
    train_df[numerical_features] = sc.fit_transform(train_df[numerical_features])
    test_df[numerical_features] = sc.transform(test_df[numerical_features])
    for col in categorical_features:
        train_df[col] = train_df[col].map(str2int_dict[col])
        test_df[col] = test_df[col].map(str2int_dict[col])
    return train_df, test_df

In [ ]:
train_df, test_df = Preprocessing(all_df)
for method in CFG.METHOD_LIST:
    gradient_boosting_model_cv_training(method, train_df, features, categorical_features)

# Inference

In [ ]:
def lightgbm_inference(x_test: pd.DataFrame):
    test_pred = np.zeros([len(x_test),4])
    for fold in range(CFG.n_folds):
        model = pickle.load(open(CFG.MODEL_DATA_PATH / f'lightgbm_fold{fold + 1}_seed{CFG.seed}_ver{CFG.VER}.pkl', 'rb'))
        # Predict
        test_pred += model.predict(x_test)
    return test_pred / CFG.n_folds

# def xgboost_inference(x_test: pd.DataFrame):
#     test_pred = np.zeros([len(x_test),4])
#     for fold in range(CFG.n_folds):
#         model = pickle.load(open(CFG.MODEL_DATA_PATH / f'xgboost_fold{fold + 1}_seed{CFG.seed}_ver{CFG.VER}.pkl', 'rb'))
#         # Predict
#         test_pred += model.predict(xgb.DMatrix(x_test), iteration_range=(0, model.best_ntree_limit))
#     return test_pred / CFG.n_folds
    
# def catboost_inference(x_test: pd.DataFrame):
#     test_pred = np.zeros([len(x_test),4])
#     for fold in range(CFG.n_folds):
#         model = pickle.load(open(CFG.MODEL_DATA_PATH / f'catboost_fold{fold + 1}_seed{CFG.seed}_ver{CFG.VER}.pkl', 'rb'))
#         # Predict
# #         print(model.predict_proba(x_test))
#         test_pred += model.predict_proba(x_test)[:,:]
#     return test_pred / CFG.n_folds

def gradient_boosting_model_inference(method: str, test_df: pd.DataFrame, features: list, categorical_features: list):
    x_test = test_df[features]
    if method == 'lightgbm':
        test_pred = lightgbm_inference(x_test)
    if method == 'xgboost':
        test_pred = xgboost_inference(x_test)
    if method == 'catboost':
        test_pred = catboost_inference(x_test)
    return test_pred

In [ ]:
for method in CFG.METHOD_LIST:
    test_pred = gradient_boosting_model_inference(method, test_df, features, categorical_features)
    test_pred_df = pd.DataFrame(test_pred, columns = [f'{method}_pred_A', f'{method}_pred_B',f'{method}_pred_D',f'{method}_pred_G'])
    test_df = pd.merge(test_df, test_pred_df, left_index=True, right_index=True)

In [ ]:
test_df['class_0'] = test_df['lightgbm_pred_A']
test_df['class_1'] = 1 - test_df['class_0']
test_df[list(submission_df)].to_csv('submission.csv', index=False)

In [ ]:
# test_df['class_1'] = 0.9 * test_df['lightgbm_pred_prob'] + 0.05 * test_df['xgboost_pred_prob'] + 0.05 * test_df['catboost_pred_prob']
# test_df['class_0'] = 1 - test_df['class_1']
# test_df[list(submission_df)].to_csv('submission.csv', index=False)

In [ ]:
test_df[list(submission_df)]